# 흉부 X-ray 폐렴 분류 — 열화 강건성 개선 실험

원본: [anjanatiha/Pneumonia-Detection-from-Chest-X-Ray-Images-with-Deep-Learning](https://github.com/anjanatiha/Pneumonia-Detection-from-Chest-X-Ray-Images-with-Deep-Learning)

런타임 → GPU(T4) 로 바꾼 뒤 위에서부터 순서대로 실행.

In [ ]:
!nvidia-smi -L
!git clone -q https://github.com/20241112-svg/Pneumonia-Detection-from-Chest-X-Ray-Images-with-Deep-Learning.git repo
%cd repo/improved
!pip install -q -r requirements.txt

## 1. Kaggle 데이터
`kaggle.json`(Kaggle → Settings → API → Create New Token) 을 업로드하세요.

In [ ]:
from google.colab import files
import os
os.makedirs('/root/.kaggle', exist_ok=True)
up = files.upload()
!mv kaggle.json /root/.kaggle/ && chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -p data/raw -q
!unzip -q -o data/raw/chest-xray-pneumonia.zip -d data/raw
!ls data/raw/chest_xray

In [ ]:
!rm -rf data/input && python make_split.py --src data/raw/chest_xray --dst data/input --val-ratio 0.1 --seed 42

## 2. 열화 샘플 확인

In [ ]:
import glob
sample = glob.glob('data/input/test/NORMAL/*')[0]
!python degrade.py --image "{sample}" --out degrade_grid.png
from IPython.display import Image; Image('degrade_grid.png', width=900)

## 3. 베이스라인 학습
원본 CNN 그대로(`base_orig`)는 TF2에서 학습이 안 됨(loss 0.693 고정, 전부 폐렴 예측). 보고서용으로 한 번만 돌리고, 이후 기준은 BN 추가판(`base`).

In [ ]:
# 원본 그대로 (학습 실패 재현, 보고서용) — 이미 한 번 돌렸으면 건너뛰어도 됨
# !python train_baseline.py --data data/input --tag base_orig --seed 0

# 수정 베이스라인 (원본 + BatchNorm)
!python train_baseline.py --data data/input --tag base --seed 0 --bn

## 4. 베이스라인 열화 강건성 — "무너지는 곡선"

In [ ]:
!python eval_robustness.py --model runs/base_s0/best.keras --data data/input --tag base_s0
from IPython.display import Image; Image('results/base_s0.png', width=700)

## 5. 개선 실험 (2주차)
한 번에 하나씩 주석 풀어서 실행. 각 실험 후 `eval_robustness.py` 로 평가.

In [ ]:
# 전처리만
!python train_baseline.py --data data/input --tag clahe --seed 0 --bn --clahe
!python eval_robustness.py --model runs/clahe_s0/best.keras --data data/input --tag clahe_s0

# 증강만 (홀드아웃 열화 제외)
# !python train_baseline.py --data data/input --tag aug --seed 0 --bn --degrade-aug
# !python eval_robustness.py --model runs/aug_s0/best.keras --data data/input --tag aug_s0

# 둘 다
# !python train_baseline.py --data data/input --tag both --seed 0 --bn --clahe --degrade-aug
# !python eval_robustness.py --model runs/both_s0/best.keras --data data/input --tag both_s0

# (선택) 백본 교체
# !python train_baseline.py --data data/input --tag effb0_both --seed 0 --backbone efficientnetb0 --clahe --degrade-aug
# !python eval_robustness.py --model runs/effb0_both_s0/best.keras --data data/input --tag effb0_both_s0

## 6. 비교 그림

In [ ]:
!python eval_robustness.py --compare results/base_s0.csv results/clahe_s0.csv --out results/compare.png
from IPython.display import Image; Image('results/compare.png', width=1000)

## 7. 결과 저장 (Drive)
런타임이 끊기면 `runs/`, `results/` 가 사라지므로 Drive 에 복사

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/pneumonia_robustness
!cp -r runs results /content/drive/MyDrive/pneumonia_robustness/
!ls /content/drive/MyDrive/pneumonia_robustness